**Défi quotidien : Comment optimiser les LLM avec LoRA**


Les méthodes d'ajustement fin à paramètres optimisés (PEFT) , comme LoRA, permettent de relever les défis de l'ajustement fin des grands modèles de langage (LLM) en ne mettant à jour qu'un petit sous-ensemble de leurs paramètres. Cette approche réduit considérablement les coûts de calcul et de stockage, rendant ainsi l'ajustement fin des LLM plus accessible. Les techniques PEFT permettent aux développeurs d'adapter des modèles pré-entraînés à des tâches spécifiques sans avoir à réentraîner l'intégralité du modèle, ce qui accélère les cycles de développement et réduit la consommation de ressources.
Vous implémenterez cette méthode pour ce défi.



👩‍🏫 👩🏿‍🏫 Ce que vous apprendrez
Comment appliquer l'adaptation de faible rang (LoRA) à un modèle de langage pré-entraîné.
Comment optimiser un modèle adapté à LoRA à l'aide de la bibliothèque Hugging Face PEFT.
Comment enregistrer et charger un modèle LoRA optimisé.
Comment effectuer une inférence à l'aide d'un modèle LoRA affiné.


🛠️ Ce que vous allez créer
Un modèle de langage finement paramétré qui génère du texte à partir d'un ensemble de données spécifique de citations, utilisant LoRA.


Ensemble de données
L'ensemble de données « Abirate/english_quotes », plus précisément un échantillon de 10 % de la partie entraînement.


Tâche
Installer les bibliothèques nécessaires (PEFT, jeux de données).
Charger un modèle de langage pré-entraîné (bigscience/bloomz-560m) et son tokenizer.
Chargez l'ensemble de données et prétraitez-le pour le modèle.
Configurez LoRA en utilisant LoraConfig.
Appliquez LoRA au modèle pré-entraîné en utilisant get_peft_model.
Configurez les arguments d'entraînement à l'aide de TrainingArguments.
Initialisez et entraînez le modèle en utilisant Trainer.
Sauvegardez le modèle LoRA optimisé.
Chargez le modèle LoRA enregistré pour l'inférence en utilisant PeftModel.from_pretrained.
Générez du texte à l'aide du modèle affiné et du tokenizer.

**Indice :**

In [ ]:
%pip install peft==0.4.0

mkdir cache

!pip install datasets

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

data = load_dataset('Abirate/english_quotes', split='train').train_test_split(test_size=0.9)['train']
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)
train_sample = data.select(range(5))
display(train_sample)

import peft
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,
    lora_alpha=1, # a scaling factor that adjusts the magnitude of the weight matrix. Usually set to 1
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none", # this specifies if the bias parameter should be trained.
    task_type="CAUSAL_LM"
)

#Add the adapter layers to the foundation model to be trained
peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())


import transformers
from transformers import TrainingArguments, Trainer
import os

output_directory = os.path.join("../cache/working", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2, # Higher learning rate than full fine-tuning.
    num_train_epochs=1,
    use_cpu=True
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
trainer.train()

import time

time_now = time.strftime("%Y-%m-%d_%H-%M-%S", time.gmtime())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

# Generate output tokens

inputs = tokenizer("Two things are infinite: ", return_tensors="pt")
outputs = peft_model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=50,
    num_return_sequences=1
    )

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))